# Ground-State Filtering for a Toy Hamiltonian

This notebook uses a bounded polynomial spectral filter as a QSVT-style surrogate for low-energy projection. The real physics task is common: given a Hamiltonian, suppress excited-state components and keep the part of a state that lies in the low-energy subspace.

The matrices are deliberately small so every step can be checked against exact diagonalization.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.design import design_projector_polynomial
from qsvt.hamiltonians import ising_hamiltonian
from qsvt.polynomials import eval_polynomial
from qsvt.rescaling import rescale_hermitian_about_cutoff
from qsvt.spectral import apply_polynomial_to_hermitian, eigh_hermitian

np.set_printoptions(precision=4, suppress=True)

## A small transverse-field Ising Hamiltonian

For two spins, use

$$H = -J Z_0 Z_1 - h(X_0 + X_1).$$

We rescale the Hamiltonian so the ground state has positive shifted energy and excited states lie on the negative side. Then a positive spectral projector polynomial selects the ground-state side of the spectrum.

In [ ]:
J = 1.0
h = 0.7

H = ising_hamiltonian(2, coupling=J, transverse_field=h)
energies, eigenvectors = eigh_hermitian(H)

energies

## Rescale around an energy cutoff

QSVT polynomials are designed on `[-1, 1]`. We map the Hamiltonian to a dimensionless operator

$$A = \frac{E_c I - H}{s},$$

so states below the cutoff have positive eigenvalues. A polynomial approximating `(1 + sign(x)) / 2` becomes a low-energy projector.

In [ ]:
cutoff = 0.5 * (energies[0] + energies[1])
scaled = rescale_hermitian_about_cutoff(H, cutoff=cutoff)
A = scaled.matrix
scale = abs(scaled.scale)
scaled_energies = np.linalg.eigvalsh(A)

cutoff, scale, scaled_energies

## Design and apply a projector polynomial

In [ ]:
degree = 17
gamma = 0.18
coeffs = design_projector_polynomial(gamma=gamma, degree=degree)

P_low = apply_polynomial_to_hermitian(A, coeffs)

ground_vector = eigenvectors[:, 0]
P_exact = np.outer(ground_vector, ground_vector.conj())

projector_error = np.linalg.norm(P_low - P_exact)
idempotence_error = np.linalg.norm(P_low @ P_low - P_low)

projector_error, idempotence_error

## Filter a trial state

A low-energy filter is useful because it increases the ground-state fraction of a state. This is the same basic spectral idea behind ground-state filtering and polynomial cooling methods.

In [ ]:
rng = np.random.default_rng(7)
psi = rng.normal(size=4)
psi = psi / np.linalg.norm(psi)

filtered = P_low @ psi
filtered = filtered / np.linalg.norm(filtered)

initial_ground_overlap = abs(np.vdot(ground_vector, psi)) ** 2
filtered_ground_overlap = abs(np.vdot(ground_vector, filtered)) ** 2

initial_ground_overlap, filtered_ground_overlap

In [ ]:
xs = np.linspace(-1, 1, 500)
ys = eval_polynomial(coeffs, xs)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, ys, label="polynomial projector")
ax.scatter(
    scaled_energies,
    eval_polynomial(coeffs, scaled_energies),
    color="tab:red",
    zorder=3,
    label="Hamiltonian spectrum",
)
ax.axvline(0.0, color="black", linewidth=1, alpha=0.4)
ax.set_xlabel("scaled eigenvalue")
ax.set_ylabel("filter value")
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.set_title("Low-energy projector polynomial")
plt.show()

## What this demonstrates

The polynomial is not a full large-scale ground-state algorithm by itself. It is the QSVT-compatible spectral primitive: after a block encoding of a rescaled Hamiltonian is available, a polynomial transform can preferentially keep low-energy components.